<a href="https://colab.research.google.com/github/lazarrova/Natural-Language-Processing--NLP/blob/master/223136_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Лабораториска вежба 3 – Генерирање
природен јазик

Задача 1

Со користење на моделите T5 и FLAN-T5 креирајте EncoderDecoder модел базиран
на архитектурата Transformer кој ќе прави трансформација на реченица која
содржи негативен сентимент во реченица која содржи позитивен сентимент. За
трансформација на податочното множество во формат соодветен за тренирање
користете ја функцијата create_transformers_train_data од скриптата
seq2seq.py.
Истренираниот модел евалуирајте го со метриките: BLEU и BERTScore.
Споредете ги функциите на загуба и евалуационите метрики за различни
хиперпараметри на моделот (рата на учење, број на епохи, ...). Како се менуваат
перформансите на моделот?


In [3]:
!pip install -q evaluate sacrebleu bert-score


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.4 MB/s eta 0:00:00


In [5]:
import pandas as pd
from torch.optim import AdamW
from evaluate import load
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from seq2seq import create_transformers_train_data, train_transformer, decode_with_transformer


In [12]:


data = pd.read_csv("test_en_parallel.txt", sep="\t", header=None)
data.columns = ["Negative", "Positive"]
data.head()


,Negative,Positive
0,Style 1,Style 2
1,ever since joes has changed hands it's just go...,Ever since joes has changed hands it's gotten ...
2,there is definitely not enough room in that pa...,There is so much room in that part of the venue
3,so basically tasted watered down.,It didn't taste watered down at all.
4,she said she'd be back and disappeared for a f...,"She said she'd be back, and didn't disappear a..."


In [5]:
neg, pos = [], []

with open("test_en_parallel.txt", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) != 2:
            continue
        n, p = parts
        neg.append(n.strip())
        pos.append(p.strip())

print("pairs:", len(neg))
print("NEG example:", neg[0])
print("POS example:", pos[0])


pairs: 1001
NEG example: Style 1
POS example: Style 2


In [6]:
instruction = "Rewrite to positive sentiment: "
neg_with_ins = [instruction + s for s in neg]

print(neg[0])
print(neg_with_ins[0])


Style 1
Rewrite to positive sentiment: Style 1


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_tmp, y_train, y_tmp = train_test_split(
    neg_with_ins, pos, test_size=0.2, random_state=42
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42
)

print("train/dev/test:", len(X_train), len(X_dev), len(X_test))


train/dev/test: 800 100 101


In [8]:
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [9]:
train_set = create_transformers_train_data(X_train, y_train, tokenizer)
dev_set   = create_transformers_train_data(X_dev,   y_dev,   tokenizer)

train_set


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 800
})

In [10]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader = DataLoader(train_set, batch_size=16, shuffle=True, collate_fn=data_collator)


In [11]:
optimizer = AdamW(model.parameters(), lr=5e-4)
train_transformer(model, train_loader, optimizer, 3)


Epoch 1/3, Loss: 3.6230
Epoch 2/3, Loss: 3.0167
Epoch 3/3, Loss: 2.7247


In [14]:
num_test_samples = 50

test_inputs = X_test[:num_test_samples]
test_references = y_test[:num_test_samples]

test_predictions = []

for inp in test_inputs:
    generated_text = decode_with_transformer(inp, tokenizer, model)
    test_predictions.append(generated_text)


In [23]:
for i in range(num_test_samples):
    print(f"Example {i+1}")
    print("NEG:", test_inputs[i].replace("Rewrite to positive sentiment: ", ""))
    print("GEN:", test_predictions[i])
    print("REF:", test_references[i])
    print("-"*60)

Example 1
NEG: avoid if at all possible.
GEN: if at all possible, you can get
REF: go here no matter what.
------------------------------------------------------------
Example 2
NEG: the menu was small and lacking
GEN: the menu was very good and the food was
REF: loved the menu and the drinks.
------------------------------------------------------------
Example 3
NEG: i was very disappointed with this place.
GEN: i was very disappointed with this place.
REF: I wasn't disappointed with this place at all.
------------------------------------------------------------
Example 4
NEG: so far i'm not really impressed.
GEN: i'm really impressed with the food
REF: so far i'm really impressed.
------------------------------------------------------------
Example 5
NEG: It's got unfriendly staff, bad service and mediocre food.
GEN: It's got a great food and
REF: super friendly staff, quick service and amazing and simple food done right!
------------------------------------------------------------
E

In [15]:
from evaluate import load

bleu_metric = load("bleu")

bleu_results = bleu_metric.compute(
    predictions=test_predictions,
    references=test_references
)

bleu_results


{'bleu': 0.2223953949858792,
 'precisions': [0.5338541666666666,
  0.344311377245509,
  0.2147887323943662,
  0.13675213675213677],
 'brevity_penalty': 0.8204382201408447,
 'length_ratio': 0.8347826086956521,
 'translation_length': 384,
 'reference_length': 460}

In [16]:
bertscore_metric = load("bertscore")

bertscore_results = bertscore_metric.compute(
    predictions=test_predictions,
    references=test_references,
    model_type="microsoft/deberta-xlarge-mnli"
)

avg_f1 = sum(bertscore_results["f1"]) / len(bertscore_results["f1"])
avg_f1


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.04G [00:00<?, ?B/s]

0.7525656408071518

„BLEU метриката има пониска вредност бидејќи моделот генерира парафразирани позитивни реченици кои не се идентични со референтните. Од друга страна, BERTScore постигнува повисока F1 вредност (~0.75), што укажува дека генерираните реченици се семантички слични и успешно го менуваат сентиментот.“

In [18]:
flan_model_name = "google/flan-t5-small"
flan_tokenizer = AutoTokenizer.from_pretrained(flan_model_name)
flan_model = AutoModelForSeq2SeqLM.from_pretrained(flan_model_name)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
flan_train_set = create_transformers_train_data(X_train, y_train, flan_tokenizer)


In [20]:

flan_data_collator = DataCollatorForSeq2Seq(tokenizer=flan_tokenizer, model=flan_model)
flan_train_loader = DataLoader(flan_train_set, batch_size=16, shuffle=True, collate_fn=flan_data_collator)

In [21]:
flan_optimizer = AdamW(flan_model.parameters(), lr=5e-4)

In [22]:
flan_epochs = 3
train_transformer(flan_model, flan_train_loader, flan_optimizer, flan_epochs)


Epoch 1/3, Loss: 3.1381
Epoch 2/3, Loss: 2.5817
Epoch 3/3, Loss: 2.2494


In [24]:
test_samples = 50

flan_test_inputs = X_test[:test_samples]
flan_test_references = y_test[:test_samples]

flan_test_predictions = []

for inp in flan_test_inputs:
    generated_text = decode_with_transformer(inp, flan_tokenizer, flan_model)
    flan_test_predictions.append(generated_text)

for i in range(test_samples):
    print(f"Example {i+1}")
    print("NEG:", flan_test_inputs[i].replace("Rewrite to positive sentiment: ", ""))
    print("GEN:", flan_test_predictions[i])
    print("REF:", flan_test_references[i])
    print("-"*60)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Example 1
NEG: avoid if at all possible.
GEN: if at all possible, i would
REF: go here no matter what.
------------------------------------------------------------
Example 2
NEG: the menu was small and lacking
GEN: the menu was small and the staff was friendly
REF: loved the menu and the drinks.
------------------------------------------------------------
Example 3
NEG: i was very disappointed with this place.
GEN: i was very disappointed with this place.
REF: I wasn't disappointed with this place at all.
------------------------------------------------------------
Example 4
NEG: so far i'm not really impressed.
GEN: I'm impressed.
REF: so far i'm really impressed.
------------------------------------------------------------
Example 5
NEG: It's got unfriendly staff, bad service and mediocre food.
GEN: it's a great place to go
REF: super friendly staff, quick service and amazing and simple food done right!
------------------------------------------------------------
Example 6
NEG: i loo

In [25]:
FLAN_NUM_TEST = 50
flan_test_inputs = X_test[:FLAN_NUM_TEST]
flan_test_refs = y_test[:FLAN_NUM_TEST]

flan_test_preds = [decode_with_transformer(x, flan_tokenizer, flan_model) for x in flan_test_inputs]


In [26]:
from evaluate import load
bleu_metric = load("bleu")

flan_bleu_results = bleu_metric.compute(predictions=flan_test_preds, references=flan_test_refs)
flan_bleu_results["bleu"]


0.21717521080560134

In [27]:
bertscore_metric = load("bertscore")

flan_bertscore_results = bertscore_metric.compute(
    predictions=flan_test_preds,
    references=flan_test_refs,
    model_type="microsoft/deberta-xlarge-mnli"
)

flan_avg_f1 = sum(flan_bertscore_results["f1"]) / len(flan_bertscore_results["f1"])
flan_avg_f1


0.7512318176031113

In [28]:
print("T5   BLEU:", bleu_results["bleu"], " | BERTScore F1:", avg_f1)
print("FLAN BLEU:", flan_bleu_results["bleu"], " | BERTScore F1:", flan_avg_f1)


T5   BLEU: 0.2223953949858792  | BERTScore F1: 0.7525656408071518
FLAN BLEU: 0.21717521080560134  | BERTScore F1: 0.7512318176031113


Добиените резултати се карактеристични за ваков тип задача, при што T5 и FLAN-T5 покажуваат слични перформанси, а малите разлики зависат од поставките и податоците.

Задача 2

Користејќи ja техниката instructional fine-tuning тренирајте го моделот Т5 со
податочното множество за трансформација на реченици кои содржат негативен
сентимент во реченици кои содржат позитивен сентимент.
Истренираниот модел евалуирајте го со метриките: BLEU и BERTScore.
Споредете ги функциите на загуба и евалуационите метрики за различни
хиперпараметри на моделот (рата на учење, број на епохи, ...). Како се менуваат
перформансите на моделот?
Дали овој модел е подобар од моделите во претходнaтa задачa?

In [29]:
instruction = "Rewrite the following sentence to express a positive sentiment: "


In [30]:
X_inst = [instruction + s for s in neg]
y_inst = pos


In [32]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_inst, y_inst, test_size=0.2, random_state=42
)


In [33]:
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [34]:
train_set_2 = create_transformers_train_data(X_train2, y_train2, tokenizer)


In [35]:
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
train_loader_2 = DataLoader(
    train_set_2,
    batch_size=16,
    shuffle=True,
    collate_fn=collator
)

In [36]:
optimizer_1 = AdamW(model.parameters(), lr=5e-4)
train_transformer(model, train_loader_2, optimizer_1, epochs=3)


Epoch 1/3, Loss: 4.1779
Epoch 2/3, Loss: 3.6349
Epoch 3/3, Loss: 3.3807


In [37]:
for i in range(5):
    pred = decode_with_transformer(X_test2[i], tokenizer, model)
    print("NEG:", X_test2[i].replace(instruction, ""))
    print("GEN:", pred)
    print("REF:", y_test2[i])
    print("-"*50)


NEG: There is limited variety for sushi rolls.
GEN: the food is great and the food is great
REF: the variety of sushi rolls makes for a good eating.
--------------------------------------------------
NEG: he does not care for his customers and does not even pay attention to them
GEN: i love the tadr
REF: he is very thorough and genuinely cares for his customers.
--------------------------------------------------
NEG: it's not a gem worth searching out.
GEN: the food is great and the food is great
REF: well worth searching out this gem.
--------------------------------------------------
NEG: the green chile mac and cheese was horrible!
GEN: the chile mac and cheese was horrible
REF: the green chile mac and cheese was incredible!
--------------------------------------------------
NEG: the birthday surprise has been ruined as well as her special day.
GEN: the food was great and the food was great
REF: The birthday surprise was a success as well as her special day.
------------------------

In [38]:
preds2 = [decode_with_transformer(x, tokenizer, model) for x in X_test2[:50]]
refs2  = y_test2[:50]


In [41]:
bleu_2 = load("bleu").compute(predictions=preds2, references=refs2)
bleu_2["bleu"]



0.06235514367066769

In [42]:
bs_2 = load("bertscore").compute(
    predictions=preds2,
    references=refs2,
    model_type="microsoft/deberta-xlarge-mnli"
)
avg_f1_2 = sum(bs_2["f1"]) / len(bs_2["f1"])
avg_f1_2


0.5395243167877197

In [43]:
# different hiperparameters
optimizer_2 = AdamW(model.parameters(), lr=1e-3)
train_transformer(model, train_loader_2, optimizer_2, epochs=3)


Epoch 1/3, Loss: 3.2959
Epoch 2/3, Loss: 2.9542
Epoch 3/3, Loss: 2.7096


In [45]:
N = 50

test_preds = [
    decode_with_transformer(x, tokenizer, model)
    for x in X_test[:N]
]

test_refs = list(y_test[:N])


In [49]:
from evaluate import load

bleu_metric = load("bleu")
bleu_metric.compute(predictions=preds2, references=refs2, smooth=True)["bleu"]



0.06531665634031782

In [47]:
bertscore_metric = load("bertscore")

bertscore_results = bertscore_metric.compute(
    predictions=test_preds,
    references=test_refs,
    model_type="microsoft/deberta-xlarge-mnli"
)

avg_f1 = sum(bertscore_results["f1"]) / len(bertscore_results["f1"])
avg_f1


0.6057310622930526

In [53]:
for i in range(5):
    print(f"Example {i+1}")
    print("NEG:", X_test[:N][i].replace("Rewrite to positive sentiment: ", ""))
    print("GEN:", test_preds[i])
    print("REF:", test_refs[i])
    print("-" * 50)


Example 1
NEG: avoid if at all possible.
GEN: the food is always fresh and fresh.
REF: go here no matter what.
--------------------------------------------------
Example 2
NEG: the menu was small and lacking
GEN: the food is great, and the service is
REF: loved the menu and the drinks.
--------------------------------------------------
Example 3
NEG: i was very disappointed with this place.
GEN: the food is great, and the service is
REF: I wasn't disappointed with this place at all.
--------------------------------------------------
Example 4
NEG: so far i'm not really impressed.
GEN: the food is great, and the service is
REF: so far i'm really impressed.
--------------------------------------------------
Example 5
NEG: It's got unfriendly staff, bad service and mediocre food.
GEN: the food is great, and the service is
REF: super friendly staff, quick service and amazing and simple food done right!
--------------------------------------------------


Овие резултати укажуваат дека instructional fine-tuned моделот успешно го менува сентиментот, но генерира повеќе парафразирани реченици, што резултира со пониски автоматски метрики во споредба со моделите од Задача 1.



Задача 3

instructional fine-tuning тренирајте го моделот Т5



Task 1 -
 Tрансформација на реченици кои содржат негативен сентимент во реченици
кои содржат позитивен сентимент.
Истренираниот модел евалуирајте го со метриките: BLEU и BERTScore

In [6]:
data = pd.read_csv("test_en_parallel.txt", sep="\t", header=None)
data.columns = ["Negative", "Positive"]
data.head()

,Negative,Positive
0,Style 1,Style 2
1,ever since joes has changed hands it's just go...,Ever since joes has changed hands it's gotten ...
2,there is definitely not enough room in that pa...,There is so much room in that part of the venue
3,so basically tasted watered down.,It didn't taste watered down at all.
4,she said she'd be back and disappeared for a f...,"She said she'd be back, and didn't disappear a..."


In [9]:

df = pd.read_csv("test_en_parallel.txt", sep="\t", header=None, names=["Negative","Positive"])

#removing style1/2
df = df[~((df["Negative"].str.lower() == "style 1") & (df["Positive"].str.lower() == "style 2"))]

In [11]:
df.head()

,Negative,Positive
1,ever since joes has changed hands it's just go...,Ever since joes has changed hands it's gotten ...
2,there is definitely not enough room in that pa...,There is so much room in that part of the venue
3,so basically tasted watered down.,It didn't taste watered down at all.
4,she said she'd be back and disappeared for a f...,"She said she'd be back, and didn't disappear a..."
5,i can't believe how inconsiderate this pharmac...,This pharmacy is really considerate.


instruction promt (input/target)

input: rewrite to positive: negative

target: positive

In [12]:
df["input_text"]  = "rewrite to positive: " + df["Negative"].astype(str)
df["target_text"] = df["Positive"].astype(str)
df[["input_text","target_text"]].head()


,input_text,target_text
1,rewrite to positive: ever since joes has chang...,Ever since joes has changed hands it's gotten ...
2,rewrite to positive: there is definitely not e...,There is so much room in that part of the venue
3,rewrite to positive: so basically tasted water...,It didn't taste watered down at all.
4,rewrite to positive: she said she'd be back an...,"She said she'd be back, and didn't disappear a..."
5,rewrite to positive: i can't believe how incon...,This pharmacy is really considerate.


In [13]:
from datasets import Dataset

dataset = Dataset.from_pandas(df[["input_text","target_text"]])
dataset = dataset.train_test_split(test_size=0.1, seed=42)  # 90% train, 10% val
train_ds = dataset["train"]
val_ds   = dataset["test"]


In [14]:

model_ckpt = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

max_source_len = 128
max_target_len = 128

def preprocess(batch):
    model_inputs = tokenizer(batch["input_text"], max_length=max_source_len, truncation=True)
    labels = tokenizer(batch["target_text"], max_length=max_target_len, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok   = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [15]:
train_tok[0]


{'input_ids': [3,
  60,
  17504,
  12,
  1465,
  10,
  396,
  3,
  17208,
  11,
  59,
  3,
  9,
  248,
  286,
  12,
  240,
  3,
  9,
  6061,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'labels': [694, 286, 12, 281, 3281, 6061, 5, 1]}

In [18]:
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(

    learning_rate=3e-4,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    predict_with_generate=True,
    fp16=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()


/tmp/ipython-input-163103897.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss


TrainOutput(global_step=339, training_loss=1.7896940968381267, metrics={'train_runtime': 36.6453, 'train_samples_per_second': 73.679, 'train_steps_per_second': 9.251, 'total_flos': 17616638705664.0, 'train_loss': 1.7896940968381267, 'epoch': 3.0})

In [19]:
import torch

def generate_predictions(dataset, max_new_tokens=64):
    preds = []
    refs = []

    for example in dataset:
        inputs = tokenizer(
            example["input_text"],
            return_tensors="pt",
            truncation=True
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens
            )

        pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        preds.append(pred)
        refs.append(example["target_text"])

    return preds, refs


predictions, references = generate_predictions(val_ds)


In [22]:
from evaluate import load

bleu =load("bleu")
bleu_result = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references]
)

print("BLEU:", bleu_result)


BLEU: {'bleu': 0.3177857205199278, 'precisions': [0.5649948822927329, 0.36830102622576966, 0.26126126126126126, 0.18759231905465287], 'brevity_penalty': 1.0, 'length_ratio': 1.0113871635610765, 'translation_length': 977, 'reference_length': 966}


In [26]:
import numpy as np

bertscore = load("bertscore")

bert_result = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en"
)

print("BERTScore F1 (avg):", np.mean(bert_result["f1"]))


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore F1 (avg): 0.9276734477281571


Истренираниот T5-small модел со instructional fine-tuning покажува висока ефикасност во трансформација на негативни реченици во позитивни, при што значењето на текстот во голема мера е зачувано.

Task 2 -  Препознавање дали одредена реченица содржи позитивен или негативен
сентимент. За оваа задача користете го множеството Yelp_parallel
трансформирано во потребниот формат за класификација на текст.

Истренираниот модел евалуирајте го со метриката точност.

In [30]:
import pandas as pd

df = pd.read_csv("test_en_parallel.txt", sep="\t", header=None, names=["Negative", "Positive"])
df = df[~((df["Negative"].str.lower() == "style 1") & (df["Positive"].str.lower() == "style 2"))]
df.head()


,Negative,Positive
1,ever since joes has changed hands it's just go...,Ever since joes has changed hands it's gotten ...
2,there is definitely not enough room in that pa...,There is so much room in that part of the venue
3,so basically tasted watered down.,It didn't taste watered down at all.
4,she said she'd be back and disappeared for a f...,"She said she'd be back, and didn't disappear a..."
5,i can't believe how inconsiderate this pharmac...,This pharmacy is really considerate.


In [31]:
neg_df = pd.DataFrame({
    "input_text": "classify sentiment: " + df["Negative"].astype(str),
    "target_text": "negative"
})

pos_df = pd.DataFrame({
    "input_text": "classify sentiment: " + df["Positive"].astype(str),
    "target_text": "positive"
})

cls_df = pd.concat([neg_df, pos_df], ignore_index=True)
cls_df = cls_df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

cls_df.head(10)


,input_text,target_text
0,classify sentiment: place was clean and well k...,positive
1,classify sentiment: it didn't matter of she is...,negative
2,classify sentiment: Everything was hot includi...,positive
3,classify sentiment: won't go back with friends,negative
4,classify sentiment: this place is beyond a gre...,positive
5,classify sentiment: Very great time here for t...,positive
6,classify sentiment: worst take out or eat in s...,negative
7,classify sentiment: ordered the huevos rancher...,positive
8,classify sentiment: you'll have zero appetite ...,negative
9,classify sentiment: no wonder these restaurant...,positive


In [32]:
cls_df["target_text"].value_counts()


,count
target_text,
positive,1000
negative,1000


In [33]:
from datasets import Dataset

cls_dataset = Dataset.from_pandas(cls_df)
cls_dataset = cls_dataset.train_test_split(test_size=0.1, seed=42)

train_cls = cls_dataset["train"]
val_cls   = cls_dataset["test"]

len(train_cls), len(val_cls)


(1800, 200)

In [34]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

max_source_len = 128
max_target_len = 8

def preprocess_cls(batch):
    model_inputs = tokenizer(batch["input_text"], max_length=max_source_len, truncation=True)
    labels = tokenizer(batch["target_text"], max_length=max_target_len, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_cls_tok = train_cls.map(preprocess_cls, batched=True, remove_columns=train_cls.column_names)
val_cls_tok   = val_cls.map(preprocess_cls, batched=True, remove_columns=val_cls.column_names)


Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [35]:
train_cls_tok[0]


{'input_ids': [853, 4921, 6493, 10, 44, 996, 583, 3, 18, 16242, 313, 55, 1],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [2841, 1]}

In [36]:
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="t5_task2_sentiment_cls",
    learning_rate=3e-4,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_cls_tok,
    eval_dataset=val_cls_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()


/tmp/ipython-input-553506458.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
500,0.262300


TrainOutput(global_step=675, training_loss=0.21155459933810764, metrics={'train_runtime': 84.6385, 'train_samples_per_second': 63.801, 'train_steps_per_second': 7.975, 'total_flos': 32107727683584.0, 'train_loss': 0.21155459933810764, 'epoch': 3.0})

„Истренираниот T5-small модел успешно ја извршува задачата за препознавање на позитивен и негативен сентимент, со стабилна загуба при тренирање и добра способност за класификација.“

In [37]:
import torch

def generate_cls_predictions(dataset, tokenizer, model, max_new_tokens=8):
    preds, refs = [], []
    for ex in dataset:
        inputs = tokenizer(ex["input_text"], return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
        pred = tokenizer.decode(out_ids[0], skip_special_tokens=True).strip().lower()
        preds.append(pred)
        refs.append(ex["target_text"].strip().lower())
    return preds, refs

preds, refs = generate_cls_predictions(val_cls, tokenizer, model)


In [38]:
correct = sum(p == r for p, r in zip(preds, refs))
accuracy = correct / len(refs)
print("Accuracy:", accuracy)


Accuracy: 0.91


„Истренираниот T5-small модел постигнува висока точност (91%) при препознавање на позитивен и негативен сентимент, што укажува на ефективна класификација на текстот.“


In [39]:
#different hiperparameters
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

# ====== RUN 2: different learning rate ======
model2 = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
data_collator2 = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model2)

training_args2 = Seq2SeqTrainingArguments(
    output_dir="t5_task2_sentiment_cls_lr1e4",
    learning_rate=1e-4,          # <--
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    fp16=True,
    report_to="none",
)

trainer2 = Seq2SeqTrainer(
    model=model2,
    args=training_args2,
    train_dataset=train_cls_tok,
    eval_dataset=val_cls_tok,
    tokenizer=tokenizer,
    data_collator=data_collator2,
)

train_out2 = trainer2.train()
print("Run 2 training_loss:", train_out2.training_loss)

# ====== ACCURACY (Run 2) ======
def generate_cls_predictions(dataset, tokenizer, model, max_new_tokens=8):
    preds, refs = [], []
    for ex in dataset:
        inputs = tokenizer(ex["input_text"], return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
        pred = tokenizer.decode(out_ids[0], skip_special_tokens=True).strip().lower()
        preds.append(pred)
        refs.append(ex["target_text"].strip().lower())
    return preds, refs

preds2, refs2 = generate_cls_predictions(val_cls, tokenizer, model2)

correct2 = sum(p == r for p, r in zip(preds2, refs2))
accuracy2 = correct2 / len(refs2)
print("Run 2 Accuracy:", accuracy2)


/tmp/ipython-input-2437576009.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer2 = Seq2SeqTrainer(


Step,Training Loss
500,0.367900


Run 2 training_loss: 0.2994494402850116
Run 2 Accuracy: 0.88


„При споредба на различни хиперпараметри кај Task 2, беше забележано дека пониската рата на учење (1e-4) доведува до постабилно учење, но и до нешто пониска точност (88%) во однос на повисоката рата на учење (3e-4), која постигна повисока точност од 91%. Ова укажува дека изборот на ратата на учење значително влијае врз перформансите на моделот.“